# TechCorp — Fine-tuning médical expérimental (QLoRA)

**Filière IA — Mission R&D**

1. Uploadez `medical_lora_train.json` depuis `rendu/data/output/` (équipe DATA)
2. Exécutez toutes les cellules
3. Notez la **loss finale**, les **epochs** et partagez le **lien Colab** dans `rapport_modele_medical.md`

In [ ]:
!pip install -q transformers==4.45.2 peft accelerate bitsandbytes datasets trl sentencepiece einops

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
DATA_PATH = "medical_lora_train.json"  # upload depuis rendu/data/output/

with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)
print(f"Exemples chargés : {len(records)}")

In [ ]:
def format_example(row):
    return (
        f"<|user|>\n{row['instruction']}\n{row['input']}<|end|>\n"
        f"<|assistant|>\n{row['output']}<|end|>"
    )

texts = [{"text": format_example(r)} for r in records]
train_ds = Dataset.from_list(texts)
train_ds[0]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./phi35_medical_lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    save_steps=200,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

history = trainer.train()
trainer.save_model("./phi35_medical_lora")
print("Loss finale:", history.training_loss)

In [ ]:
TEST_PROMPTS = [
    "Quels sont les symptômes courants de l'hypertension ?",
    "Puis-je arrêter mes antibiotiques si je me sens mieux ?",
    "Quelle est la différence entre une grippe et un rhume ?",
]

model.eval()
for prompt in TEST_PROMPTS:
    formatted = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.2, do_sample=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print("=" * 60)
    print("Q:", prompt)
    print("R:", text.split("<|assistant|>")[-1][:500])